In [ ]:
import pandas as pd
# 외부 데이터프레임 호출
fish = pd.read_csv('https://bit.ly/fish_csv_data')
fish.head()

fish_input = fish[['Weight', 'Length', 'Diagonal', 'Height', 'Width']]
fish_target = fish['Species']
fish_input.head()



# 훈련-테스트 데이터(입력/타깃) 추출
from sklearn.model_selection import train_test_split
train_input, test_input, train_target, test_target = train_test_split(fish_input, fish_target, random_state=42)



from sklearn.preprocessing import StandardScaler
# 데이터 전처리 - 표준화
ss = StandardScaler()
ss.fit(train_input)
train_scaled = ss.transform(train_input)
test_scaled = ss.transform(test_input)



#K-NN 분류 모델 생성
from sklearn.neighbors import KNeighborsClassifier
kn = KNeighborsClassifier(n_neighbors=3)
kn.fit(train_scaled, train_target)
print("K-NN 훈련 정확도: ", kn.score(train_scaled, train_target))
print("K-NN 테스트 정확도: ", kn.score(test_scaled, test_target))
print("K-NN 클래스: ", kn.classes_)
print(kn.predict(test_scaled[:5]))


import numpy as np
# 처음 5개 테스트 입력 데이터에 대한 타깃 예측 + 확률 출력
proba = kn.predict_proba(test_scaled[:5])
print(np.round(proba, decimals=4))

# kn.neighbors는 2차원 배열 입력 기대 => Numpy 배열의 슬라이싱 연산자는 항상 2차원 배열 출력 => indexes도 2차원 => indexes에 저장된 번호 활용 시 indexes[0]
distances, indexes = kn.kneighbors(test_scaled[3:4])
print("indexes 출력: ", indexes)
print(train_target.iloc[indexes[0]])



# 로지스틱(시그모이드) 함수
import numpy as np
import matplotlib.pyplot as plt

z = np.arange(-5, 5, 0.1)
phi = 1 / (1 + np.exp(-z))
plt.plot(z, phi)
plt.xlabel('z')
plt.ylabel('phi')
plt.show()



# boolean indexing 예시(이진 분류에 활용)
char_arr = np.array(['A', 'B', 'C', 'D', 'E'])
print(char_arr[[True, False, True, False, False]]) # 대괄호 2개!!

# 로지스틱 회귀를 활용한 이진 분류(도미(bream)/빙어(smelt))
bream_smelt_indexes = (train_target == 'Bream') | (train_target == 'Smelt') #bream_smelt_indexes는 불리언 배열(True or False 저장)
train_bream_smelt = train_scaled[bream_smelt_indexes]
target_bream_smelt = train_target[bream_smelt_indexes]

from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()

lr.fit(train_bream_smelt, target_bream_smelt)
print(lr.predict(train_bream_smelt[:5]))
print(lr.predict_proba(train_bream_smelt[:5])) # 첫 번째 열 - 음성 클래스(도미)& 두 번째 열 - 양성 클래스(빙어)
print(lr.classes_)
print(lr.coef_, lr.intercept_)  # 이진분류에서 z는 유일하다(양성 클래스 기준), z를 결정하는 a,b,c,d,e + f(절편) 출력

decisions = lr.decision_function(train_bream_smelt[:5])
print(decisions) # 선형 방정식에 따른 z값 출력

from scipy.special import expit # scipy 라이브러리 => 로지스틱(시그모이드) 함수 호출
print(expit(decisions)) # 양성 클래스 z값에 따른 확률 = 양성 클래스일 확률 = lr.predict_proba(train_bream_smelt[:5])의 두 번째 열과 같다



#로지스틱 회귀를 활용한 다중 분류
lr = LogisticRegression(C = 20, max_iter = 1000)
lr.fit(train_scaled, train_target)
# 준수한 훈련-테스트 정확도 확인
print(lr.score(train_scaled, train_target))
print(lr.score(test_scaled, test_target))

# 상위 5개 샘플 & 클래스 7개 => 5행 7열의 proba 배열(각 행이 클래스 7개 중 하나일 확률 나열)
proba = lr.predict_proba(test_scaled[:5])
print(np.round(proba, decimals=3))
print(lr.classes_)
# 다중분류 로지스틱 회귀의 가중치 크기: 7행 5열 => 7행? 클래스가 7개! 5열? 가중치가 5개!(a, b, c, d, e에 매칭되는 특성)
# 즉, 다중분류 로지스틱 회귀는 클래스 개수만큼 z값을 만들어낸다!(클래스 7개 => z값도 7개)
print(lr.coef_.shape, lr.intercept_.shape)

decisions = lr.decision_function(test_scaled[:5])
print(np.round(decisions, decimals = 2))

# 7개 z 값에 따른 샘플별 7개 클래스 확률 = lr.predict_proba(test_scaled[:5])와 같다
from scipy.special import softmax
proba = softmax(decisions, axis = 1)
print(np.round(proba, decimals = 3))